[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jman4162/sensortwin-transformer-agent/blob/master/notebooks/07_robustness_colab.ipynb)

# SensorTwin — robustness, domain shift, and calibration under corruption

Runs `scripts/robustness_report` (same as `make robustness-full`): trains XGBoost / CNN /
transformer / pretrained-transformer on domain A with the committed parity recipes, then measures
degradation under Gaussian noise, shorter windows, missing channels, and a shifted measurement
regime (domain B), plus ECE before/after temperature scaling.

Two arms: `augmented` (the shared recipe includes channel-dropout augmentation — comparable across
models, but flattering on the missing-channel probe) and `no_augment` (separates "robust" from
"trained on the probe"). Budget: ~4 trainings per (arm, seed); expect an afternoon on a T4 for
3 seeds x both arms. Per-(arm, seed) checkpoints make an interrupted run resumable.

In [ ]:
# Opened from the Colab badge? Only the notebook is present — clone the public repo, then install.
import os

if not os.path.exists("sensortwin"):
    !git clone https://github.com/jman4162/sensortwin-transformer-agent.git
    %cd sensortwin-transformer-agent
%pip install -q -e ".[ml]"

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
SEEDS = [0, 1, 2]
EPOCHS = 30
ARM = "both"         # augmented | no_augment | both
OUT = "reports/experiment_summaries"

In [ ]:
from scripts.robustness_report import main as robustness

robustness([
    "--mode", "colab_standard",
    "--epochs", str(EPOCHS),
    "--seeds", *[str(s) for s in SEEDS],
    "--arm", ARM,
    "--out", OUT,
])

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

display(Markdown(Path(f"{OUT}/robustness_summary.md").read_text()))

In [ ]:
try:
    from google.colab import files

    files.download(f"{OUT}/robustness_summary.md")
    files.download(f"{OUT}/robustness_summary.json")
except Exception as e:
    print("Not in Colab or download unavailable:", e)

## Reading the result

- Compare the Missing-ch Δ column across arms: the `augmented` arm trains on the probe, so its
  smaller deltas measure recipe + architecture together; the `no_augment` arm isolates the
  architecture.
- The pretrained-vs-scratch shift gap tests whether masked pretraining buys robustness or just
  memorized the generator.
- Commit `robustness_summary.{json,md}` so the model card §8 traces to this run.